# Week 4 — profile robustness and selection

This notebook brings the three completed Week 4 analyses into one sequence.
The population, feature definitions, weighting, fitting procedures and decision
criteria are unchanged. Shared data and functions are defined once.

## Contents

1. Shared data, feature geometry and execution settings.
2. K-means representations, reference fits, sampling and stability screening.
3. GMM, Ward, average linkage and DBSCAN: implementation, failures and screening.
4. A–F membership comparisons, explanations and the B/E decision.

Run from top to bottom in a fresh Python kernel. Keep the refit switches off
to read the saved full experiments and reproduce the displayed checks. Enabling
a refit writes comparison outputs, not replacement frozen models.

This notebook needs the course source/prepared data and the referenced saved
model/experiment files. It does not recreate every historical search from the
raw CSV alone. The original full run histories remain archived; no new clustering
or model-selection experiment is introduced by combining these files.

The three preceding notebooks are preserved here:
K-means (historical notebook retained in the course archive),
alternative methods (historical notebook retained in the course archive),
partition comparisons (historical notebook retained in the course archive).

## 1. Shared data and setup

Use Python 3.11 and `requirements-objective1.txt`. Run from the course folder
or `weekly_notebooks`, with `23_wrp` and `course_artifacts` available in the
course folder. Only load trusted local course `.joblib` files.

In [ ]:
from pathlib import Path
from itertools import combinations
import hashlib
import json
import joblib
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.cluster import KMeans
from sklearn.metrics import adjusted_rand_score
from threadpoolctl import threadpool_limits
from scipy.spatial.distance import cdist
from sklearn.mixture import GaussianMixture
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors
from scipy.cluster.hierarchy import linkage, cut_tree
from scipy.sparse import vstack
from scipy.optimize import linear_sum_assignment
from sklearn.impute import SimpleImputer
from sklearn.model_selection import GroupKFold
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import balanced_accuracy_score, precision_score, recall_score, f1_score

### Optional refits — all off by default

In [ ]:
RUN_REFERENCES = False  # 81 reference fits, 20 initializations each.
RUN_STABILITY = False  # 8,100 fits plus 198,450 full-population pairwise ARIs.
RUN_NEW_REFERENCE = False  # One optional fit, not an implicit rerun of the full search.
CANDIDATE = 170
RUN_EXPLANATIONS = False

### 1.1. Keep the population and feature geometry explicit

    This is the completed instructor search, not only Moodle Experiment 1.
    Experiment 1 concerned combined versus separate harm under balanced 0/1/2
    worry coding, k = 2,...,10. Do not treat the later three-coding/three-allocation
    grid as that single experiment. The archived notebook preserves earlier work.

    The final search uses 109,488 complete-response development respondents and
    separates personal and known-person harm. No imputation or z-standardization.
    For worry values x in {0,1,2}, replace 2 with t and divide by t. Use t = 2,3,5.
    Let m be the respondent's mean across the six recoded worry values.
    The 21 coordinates are
    $[(x_j-m)\sqrt{\alpha/6},\;m\sqrt{\alpha/6},\;h_l\sqrt{(1-\alpha)/14}]$.
    Thus the mean-worry coordinate is deliberately reduced, not the usual
    unmodified six-coordinate worry block. Alpha is .50, .75 or .25.

In [ ]:
# Run from the course folder or weekly_notebooks. No absolute data paths.
ROOT = Path.cwd()
while not (ROOT / 'course_artifacts').is_dir() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
assert (ROOT / 'course_artifacts').is_dir(), 'Open the course folder first.'
ART = ROOT / 'course_artifacts'
OUT = ART / 'objective1_simplified_checks'
OUT.mkdir(exist_ok=True)


In [ ]:
DOMAINS = ['food', 'water', 'crime', 'severe_weather', 'traffic', 'mental_health']
WORRY = ['worry_' + d for d in DOMAINS]
HARM = ['harm_' + d for d in DOMAINS + ['work']]
SELF = [f + '_personal' for f in HARM]
KNOWN = [f + '_known_person' for f in HARM]
FEATURES = WORRY + SELF + KNOWN


In [ ]:
def country_weights(frame):
    # Each country has equal total mass; preserve relative survey weights within it.
    return (frame.WGT / frame.groupby('COUNTRY_ISO3').WGT.transform('sum')
            / frame.COUNTRY_ISO3.nunique()).to_numpy()


In [ ]:
SOURCE = ART / 'week04_search_v4/stage1_eligible_development.parquet'
SAVED = ART / 'week04_experiment1_complete_overall_worry_v3'
source = pd.read_parquet(SOURCE)
assert source.country_split.eq('train').all()
data = source.loc[source[FEATURES].notna().all(axis=1)].reset_index(drop=True)
weights = country_weights(data)
assert len(data) == 109488 and data.COUNTRY_ISO3.nunique() == 113
assert data.WPID_RANDOM.is_unique
design = json.loads((SAVED / 'design.json').read_text())
fingerprint = hashlib.sha256(SOURCE.read_bytes() +
                            json.dumps(design, sort_keys=True).encode()).hexdigest()


In [ ]:
def profile_matrix(frame, top, alpha):
    values = frame[FEATURES].to_numpy(float)
    assert np.isfinite(values).all(), 'This analysis uses complete responses only.'
    worry = np.where(values[:, :6] == 2, top, values[:, :6]) / top
    mean_worry = worry.mean(axis=1, keepdims=True)
    # Preserve the original floating-point operation order for cache comparisons.
    return np.column_stack([
        (worry - mean_worry) * np.sqrt(alpha) / np.sqrt(6),
        mean_worry * np.sqrt(alpha) / np.sqrt(6),
        values[:, 6:] * np.sqrt((1 - alpha) / 2) / np.sqrt(7),
    ])


## 2. K-means representations and stability

In [ ]:
ENCODINGS = {'012': 2., '013': 3., '015': 5.}
BALANCES = {'balanced': .5, 'worry_emphasized': .75, 'harm_emphasized': .25}
references = {}
for encoding, top in ENCODINGS.items():
    for balance, alpha in BALANCES.items():
        X = profile_matrix(data, top, alpha)
        for k in range(2, 11):
            saved = joblib.load(SAVED / f'reference_{encoding}_{balance}_k{k}.joblib')
            assert saved['fingerprint'] == fingerprint
            references[(encoding, balance, k)] = saved
            if RUN_REFERENCES:
                with threadpool_limits(limits=1):
                    model = KMeans(n_clusters=k, n_init=20, random_state=20231006)
                    model.fit(X, sample_weight=weights)
                joblib.dump({'fingerprint': fingerprint, 'labels': model.labels_,
                             'centres': model.cluster_centers_},
                            OUT / f'reference_{encoding}_{balance}_k{k}.joblib')
print('Verified cached reference fits:', len(references))


### 2.2. Use the same 50 respondent and 50 country samples for every candidate

    Respondent test: retain 80% without replacement within each country.
    Country test: retain 90 development countries, stratified by region × income.
    At least one country per stratum remains. Singleton strata are always included;
    the country test does not assess their omission. Read the fixed plans and
    verify them below. Do not generate a different sample for each model.
    Each fitted model assigns the same complete development population.

In [ ]:
plans = joblib.load(SAVED / 'sampling_indices.joblib')
display(pd.read_csv(SAVED / 'stratum_quotas.csv'))
display(pd.read_csv(SAVED / 'sampling_plans.csv').groupby('experiment').agg(
    repetitions=('repeat', 'count'), respondents_min=('respondents', 'min'),
    respondents_max=('respondents', 'max'), countries=('countries', 'min')))
assert set(plans) == {'respondents', 'countries'}
for kind, repetitions in plans.items():
    assert len(repetitions) == 50
    for indices in repetitions:
        assert len(indices) == len(np.unique(indices))
        assert indices.min() >= 0 and indices.max() < len(data)


In [ ]:
# The small sampling routine below reproduces the saved plan exactly.
registry = pd.read_csv(ART / 'week03_country_split.csv')
registry = registry.loc[registry.country_split.eq('train')].copy()
groups = registry.groupby(['GlobalRegion', 'CountryIncomeLevel'], sort=True)
quota = groups.size().rename('available').to_frame()
quota['selected'] = np.maximum(1, np.floor(.8 * quota.available).astype(int))
while quota.selected.sum() < round(.8 * len(registry)):
    remainder = (.8 * quota.available - quota.selected).where(quota.selected < quota.available, -np.inf)
    quota.loc[remainder.idxmax(), 'selected'] += 1
country_rows = [np.flatnonzero(data.COUNTRY_ISO3.eq(c)) for c in sorted(data.COUNTRY_ISO3.unique())]
for repeat in range(50):
    rng = np.random.default_rng(20261005 + repeat)
    respondents = np.sort(np.concatenate([
        rng.choice(rows, round(.8 * len(rows)), replace=False) for rows in country_rows]))
    selected_countries = []
    for key, group in groups:
        selected_countries.extend(rng.choice(sorted(group.COUNTRY_ISO3),
                                             int(quota.loc[key, 'selected']), replace=False))
    countries = np.flatnonzero(data.COUNTRY_ISO3.isin(selected_countries))
    assert np.array_equal(respondents, plans['respondents'][repeat])
    assert np.array_equal(countries, plans['countries'][repeat])
print('All 100 sampling plans reproduced exactly.')


### 2.3. Refit and compare partitions only when explicitly requested

    All 50 choose 2 = 1,225 unweighted ARIs are used per candidate per test.
    Retention requires median and lower-decile ARI ≥ .80 in both tests.
    Shared-training-country ARI is a separate diagnostic, not the retention score.
    Existing fit fingerprints are checked before reuse. New fits use a separate
    output directory. The default does not repeat thousands of completed fits.

In [ ]:
if RUN_STABILITY:
    rows = []
    for (encoding, balance, k), reference in references.items():
        X = profile_matrix(data, ENCODINGS[encoding], BALANCES[balance])
        for kind, samples in plans.items():
            partitions = []
            for repeat, indices in enumerate(samples):
                cache = SAVED / 'fits' / f'{encoding}_{kind}_{repeat:02d}_{balance}_k{k}.joblib'
                result = joblib.load(cache) if cache.exists() else {}
                if result.get('fingerprint') != fingerprint:
                    with threadpool_limits(limits=1):
                        model = KMeans(n_clusters=k, n_init=20, random_state=20261005 + 1000 + repeat)
                        model.fit(X[indices], sample_weight=country_weights(data.iloc[indices]))
                        result = {'fingerprint': fingerprint, 'labels': model.predict(X)}
                    joblib.dump(result, OUT / cache.name)
                partitions.append(result['labels'])
            ari = [adjusted_rand_score(a, b) for a, b in combinations(partitions, 2)]
            rows.append({'encoding': encoding, 'balance': balance, 'k': k, 'test': kind,
                         'median_ari': np.median(ari), 'p10_ari': np.quantile(ari, .1)})
    pd.DataFrame(rows).to_csv(OUT / 'week04_recomputed_stability.csv', index=False)


In [ ]:
kmeans_screen = pd.read_csv(SAVED / 'candidate_comparison.csv', dtype={'encoding': str})
kmeans_screen['dominance'] = kmeans_screen.k * kmeans_screen.largest
ari_columns = ['median_pairwise_country', 'p10_pairwise_country',
               'median_pairwise_respondent', 'p10_pairwise_respondent']
kmeans_screen['stable'] = kmeans_screen[ari_columns].ge(.8).all(axis=1)
kmeans_screen['structural_pass'] = (kmeans_screen.k.ge(3) & kmeans_screen.largest.le(.4)
                                  & kmeans_screen.dominance.lt(3))
retained_kmeans = kmeans_screen.loc[kmeans_screen.stable & kmeans_screen.structural_pass].copy()
retained_kmeans.insert(0, 'model', list('ABCDEF'))
assert len(retained_kmeans) == 6
display(kmeans_screen[['encoding', 'balance', 'k', 'stable', 'structural_pass']])
display(retained_kmeans[['model', 'encoding', 'balance', 'k'] + ari_columns + ['largest', 'dominance']].round(3))


### 2.4. Inspect original responses, not only numerical separation

    Describe each cluster by original worry-category proportions and personal/
    known-person harm prevalence. These six A–F partitions meet numerical filters;
    that alone does not make six different substantive findings. Compare their
    overlap and simple-feature explanations in Section 4.

In [ ]:
display(pd.read_csv(SAVED / 'profile_descriptions_all_candidates.csv').head(30))
fig, axes = plt.subplots(1, 2, figsize=(12, 3.5))
for ax, name, key in zip(axes, ['B: 0/1/2, balanced, k=4', 'E: 0/1/3, worry emphasis, k=8'],
                         [('012', 'balanced', 4), ('013', 'worry_emphasized', 8)]):
    kmeans_labels = references[key]['labels']
    shares = np.bincount(kmeans_labels, weights=weights, minlength=key[2])
    ax.bar(range(key[2]), 100 * shares)
    ax.set(title=name, xlabel='Fixed cluster ID', ylabel='Equal-country share (%)')
plt.tight_layout()
plt.show()


### 2.5. Result of the numerical screen

A–F meet the numerical filters. This is a candidate set, not yet the final
choice. Sections 3 and 4 inspect method agreement, substantive meaning,
membership overlap and simple explanations before retaining B/E.

## 3. Alternative clustering methods

### 3.1. Use the same complete-response population and feature geometry

    K-means is the weighted reference. GMM and hierarchical clustering do not
    directly accept the same respondent weights. Approximate them by drawing an
    equal number of respondents per country, with replacement and probabilities
    proportional to the survey weights. This changes the fitting procedure and
    must remain visible in the interpretation.

In [ ]:
ALT = ART / 'week04_other_algorithms_v1'
alternative_design = json.loads((ALT / 'design.json').read_text())
alternative_fingerprint = hashlib.sha256(SOURCE.read_bytes() +
    json.dumps(alternative_design, sort_keys=True).encode()).hexdigest()
saved_specs = joblib.load(ALT / 'specifications.joblib')
assert saved_specs['fingerprint'] == alternative_fingerprint
alternative_specs = saved_specs['specs']
# Preserve the calibrated DBSCAN radius grid, including skipped zero radii.
display(pd.read_csv(ALT / 'dbscan_radius_grid.csv').head(12))
alternative_screen = pd.read_csv(ALT / 'reference_screen.csv', dtype={'encoding': str})
display(alternative_screen.groupby(['algorithm', 'status']).size().rename('reference_fits'))


In [ ]:
def country_bootstrap(training, per_country, seed):
    rng = np.random.default_rng(seed)
    return np.concatenate([
        rng.choice(group.index, per_country, replace=True, p=group.WGT / group.WGT.sum())
        for _, group in training.groupby('COUNTRY_ISO3', sort=True)])


### 3.2. Declare the models directly

    GMM uses diagonal or full covariance, five initializations and a covariance
    floor of .001. It fits an approximately full-size equal-country bootstrap.

    Ward and average linkage use 150 draws per country: 16,950 rows for the full
    reference. Cluster that bootstrap, calculate its cluster centroids, then
    assign every complete respondent to the nearest centroid. Duplicated draws
    carry mass in the centroids. This centroid extension is the adopted algorithm,
    not native full-population Ward. It also provides out-of-sample assignment.
    The condensed distance array alone grows quadratically; guard its size.

    DBSCAN fits distinct response patterns with equal-country sample mass. Build
    a sparse radius graph and stop above five million edges. Extend using the
    nearest core pattern within epsilon, but preserve native training labels.
    Failures are recorded, not silently converted into poor stability scores.

In [ ]:
def fit_gmm(X, training, k, covariance, seed):
    per_country = max(1, round(len(training) / training.COUNTRY_ISO3.nunique()))
    draw = country_bootstrap(training, per_country, seed)
    model = GaussianMixture(n_components=k, covariance_type=covariance,
                            n_init=5, max_iter=300, reg_covar=.001, random_state=seed)
    with threadpool_limits(limits=1):
        model.fit(X[draw])
        if not model.converged_:
            raise RuntimeError('GMM did not converge.')
        return model.predict(X)


In [ ]:
def fit_hierarchical(X, training, k, method, seed):
    draw = country_bootstrap(training, 150, seed)
    if len(draw) * (len(draw) - 1) // 2 * 8 > 2_000_000_000:
        raise MemoryError('Hierarchical distance array exceeds the 2 GB guard.')
    with threadpool_limits(limits=1):
        tree = linkage(X[draw], method=method, metric='euclidean')
        native = cut_tree(tree, n_clusters=[k]).ravel()
        centres = np.array([X[draw][native == j].mean(axis=0) for j in range(k)])
        return cdist(X, centres).argmin(axis=1)


In [ ]:
def fit_dbscan(X, training, eps, min_samples):
    with threadpool_limits(limits=1):
        patterns, inverse = np.unique(X[training.index], axis=0, return_inverse=True)
        mass = np.bincount(inverse, weights=country_weights(training) * len(X))
        neighbours = NearestNeighbors(radius=eps, n_jobs=1).fit(patterns)
        blocks, edges = [], 0
        for start in range(0, len(patterns), 250):
            block = neighbours.radius_neighbors_graph(patterns[start:start + 250], mode='distance')
            edges += block.nnz
            if edges > 5_000_000:
                raise MemoryError('DBSCAN radius graph exceeds five million edges.')
            blocks.append(block)
        graph = vstack(blocks, format='csr')
        graph.setdiag(0.)
        model = DBSCAN(eps=eps, min_samples=min_samples, metric='precomputed')
        native = model.fit_predict(graph, sample_weight=mass)
        labels = np.full(len(X), -1, dtype=int)
        core = model.core_sample_indices_
        if len(core):
            nearest = NearestNeighbors(n_neighbors=1, n_jobs=1).fit(patterns[core])
            distance, index = nearest.kneighbors(X)
            covered = distance[:, 0] <= eps
            labels[covered] = native[core[index[covered, 0]]]
        labels[training.index] = native[inverse]
        return labels


In [ ]:
if RUN_NEW_REFERENCE:
    spec = alternative_specs[CANDIDATE]
    top = {'012': 2., '013': 3., '015': 5.}[spec['encoding']]
    alpha = {'balanced': .5, 'worry_emphasized': .75, 'harm_emphasized': .25}[spec['balance']]
    X = profile_matrix(data, top, alpha)
    try:
        if spec['algorithm'] == 'gmm':
            labels = fit_gmm(X, data, spec['k'], spec['covariance'], seed=20231006)
        elif spec['algorithm'] == 'agglomerative':
            labels = fit_hierarchical(X, data, spec['k'], spec['linkage'], seed=20231006)
        else:
            labels = fit_dbscan(X, data, spec['eps'], spec['min_samples'])
        result = {'status': 'ok', 'labels': labels, 'spec': spec}
    except (MemoryError, ValueError, RuntimeError) as error:
        result = {'status': 'failed', 'error': str(error), 'spec': spec}
    joblib.dump(result, OUT / f'alternative_reference_{CANDIDATE}.joblib')


### 3.3. Structural screening, then both stability tests

    Reference screening: at least three clusters, largest assigned cluster ≤40%,
    dominance k × largest share <3, and assigned coverage ≥90% for DBSCAN.
    Only structurally eligible references enter 50 respondent and 50 country
    repetitions. Both median and lower-decile ARI must be ≥.80 in both tests.
    DBSCAN ARI uses respondents assigned by both fits; low joint coverage cannot
    pass as high stability. Non-convergence and graph-memory guards remain failures.
    The archived notebook contains the full grid/repetition orchestration;
    this notebook keeps the model code and reads its checked completed evidence.

In [ ]:
alternative_stability = pd.read_csv(ALT / 'stability_summary.csv')
eligible = alternative_screen.loc[alternative_screen.structural_pass].copy()
passes = alternative_stability.groupby('candidate').agg(
    tests=('experiment', 'nunique'), all_pass=('stability_pass', 'all'),
    fits_min=('successful_fits', 'min'), complete=('complete_pairs', 'all'))
retained_alternatives = passes.index[(passes.tests == 2) & passes.all_pass & passes.fits_min.eq(50) & passes.complete]
display(alternative_screen.groupby('algorithm').agg(attempted=('candidate', 'size'),
    successful=('status', lambda x: x.eq('ok').sum()), structurally_eligible=('structural_pass', 'sum')))
display(alternative_screen.loc[alternative_screen.status.ne('ok'), ['candidate', 'algorithm', 'error']])
display(alternative_stability.loc[alternative_stability.candidate.isin(retained_alternatives)].round(3))
assert list(retained_alternatives) == [170]
display(pd.read_csv(ALT / 'B_E_comparisons.csv').round(3))


### 3.4. What the alternative methods add

    Only Ward candidate 170 (0/1/3, worry emphasis, k = 8) passes both screens.
    Its centroid-extended labels agree substantially with K-means E8 (ARI about
    .820; matched weighted agreement about 82%). GMM and average linkage add no
    retained solution. DBSCAN yields no structurally eligible reference; 23
    reference attempts stop at the graph-memory guard. These are not 23 findings
    that no density structure exists.

    Agreement between Ward and K-means supports E8, but both use squared-Euclidean
    compactness and the same representation. It is not independent confirmation
    of a unique social taxonomy. Keep the simpler directly weighted K-means B/E
    definitions for the frozen handoff; use Ward as a robustness comparison.

## 4. Compare the retained partitions and select B/E

### 4.1. Recover fixed A–F labels; do not fit another clustering

    A = 0/1/2, worry emphasis, k3; B = 0/1/2, balanced, k4;
    C/D/E = 0/1/3, worry emphasis, k6/k7/k8;
    F = 0/1/5, worry emphasis, k6. All six meet the numerical screen.

In [ ]:
partition_specs = {'A': ('012', 'worry_emphasized', 3), 'B': ('012', 'balanced', 4),
                  'C': ('013', 'worry_emphasized', 6), 'D': ('013', 'worry_emphasized', 7),
                  'E': ('013', 'worry_emphasized', 8), 'F': ('015', 'worry_emphasized', 6)}
partition_labels = {}
for name, (encoding, balance, k) in partition_specs.items():
    partition_labels[name] = references[(encoding, balance, k)]['labels']


### 4.2. Compare memberships and approximate refinement

    ARI measures unweighted partition agreement, independent of cluster IDs.
    Weighted overlaps show which groups correspond. For coarse/fine pairs,
    refinement purity is the sum of each fine cluster's largest overlap with a
    coarse cluster; also report the worst fine-cluster purity. A good average
    alone can hide a badly split small group. This is an approximate graph,
    not a mathematical lattice or evidence of transitivity.

In [ ]:
rows = []
partition_overlaps = {}
for first, second in combinations(partition_labels, 2):
    a, b = partition_labels[first], partition_labels[second]
    table = pd.crosstab(a, b, values=weights, aggfunc='sum').fillna(0)
    partition_overlaps[(first, second)] = table
    fine_purity = table.max(axis=0) / table.sum(axis=0)
    row, column = linear_sum_assignment(-table.to_numpy())
    rows.append({'coarse': first, 'fine': second, 'ARI': adjusted_rand_score(a, b),
                 'refinement_purity': table.max(axis=0).sum(),
                 'worst_fine_purity': fine_purity.min(),
                 'matched_weight': table.to_numpy()[row, column].sum() if table.shape[0] == table.shape[1] else np.nan})
partition_comparison = pd.DataFrame(rows)
display(partition_comparison.round(3))
display(partition_overlaps[('B', 'E')].round(3))
partition_comparison.to_csv(OUT / 'week04_recomputed_partition_comparisons.csv', index=False)
plt.figure(figsize=(8, 3))
plt.imshow(partition_overlaps[('B', 'E')], aspect='auto', cmap='Blues')
plt.xticks(range(8), range(8)); plt.yticks(range(4), range(4))
plt.xlabel('E cluster ID'); plt.ylabel('B cluster ID')
plt.title('B/E membership overlap: equal-country mass')
plt.colorbar(label='Weighted share of all respondents')
plt.tight_layout(); plt.show()


### 4.3. Check whether a simple burden score or one feature explains the labels

    Use the same five country-grouped folds as Week 3. These models predict fixed
    cluster labels, not new clusterings. Depth-1/2/3/5 trees use mean original
    worry and mean combined harm (not fourteen duplicated harm entries).
    The single-feature rules use all twenty original separate-harm features;
    the saved tables also retain the thirteen-feature combined-harm companion.
    High recall alone is not a cluster definition: also inspect precision,
    balanced accuracy, outside prevalence and coverage.

In [ ]:
# The burden-tree function below recalculates equal-country weights within each country fold.
def burden_folds(frame, labels, weights, worry, harm, depths, seed):
    scores = pd.DataFrame({'worry': frame[worry].mean(axis=1),
                           'harm': frame[harm].mean(axis=1)})
    rows = []
    cv = GroupKFold(n_splits=5)
    for fold, (train, test) in enumerate(cv.split(scores, groups=frame.COUNTRY_ISO3), 1):
        train_weight = country_weights(frame.iloc[train])
        test_weight = country_weights(frame.iloc[test])
        imputer = SimpleImputer(strategy='median')
        x_train = imputer.fit_transform(scores.iloc[train])
        x_test = imputer.transform(scores.iloc[test])
        for depth in depths:
            tree = DecisionTreeClassifier(max_depth=depth, min_samples_leaf=200,
                                          random_state=seed)
            tree.fit(x_train, labels[train], sample_weight=train_weight)
            prediction = tree.predict(x_test)
            rows.append({'fold': fold, 'depth': depth,
                         'balanced_accuracy': balanced_accuracy_score(
                             labels[test], prediction, sample_weight=test_weight),
                         'macro_f1': f1_score(labels[test], prediction, average='macro',
                                              sample_weight=test_weight, zero_division=0)})
    return pd.DataFrame(rows)


In [ ]:
def one_feature_folds(frame, labels, weights, features):
    # Search balanced-accuracy rules, not a Gini-optimized sklearn stump.
    rows = []
    cv = GroupKFold(n_splits=5)
    for fold, (train, test) in enumerate(cv.split(frame, groups=frame.COUNTRY_ISO3), 1):
        for cluster in np.unique(labels):
            target = labels == cluster
            best_score, best_rule = -np.inf, None
            for feature in features:
                x = frame[feature].to_numpy(float)
                observed = train[np.isfinite(x[train])]
                categories = np.unique(x[observed])
                if len(categories) < 2 or np.unique(target[observed]).size < 2:
                    continue
                for threshold in (categories[:-1] + categories[1:]) / 2:
                    for greater in [True, False]:
                        prediction = x[observed] > threshold if greater else x[observed] <= threshold
                        score = balanced_accuracy_score(target[observed], prediction,
                                                        sample_weight=weights[observed])
                        if score > best_score:
                            best_score, best_rule = score, (feature, threshold, greater)
            if best_rule is None:
                continue
            feature, threshold, greater = best_rule
            x = frame[feature].to_numpy(float)
            observed = test[np.isfinite(x[test])]
            prediction = x[observed] > threshold if greater else x[observed] <= threshold
            rows.append({'fold': fold, 'cluster': int(cluster), 'feature': feature,
                         'threshold': threshold, 'greater': greater,
                         'balanced_accuracy': balanced_accuracy_score(
                             target[observed], prediction, sample_weight=weights[observed]),
                         'precision': precision_score(target[observed], prediction,
                             sample_weight=weights[observed], zero_division=np.nan),
                         'recall': recall_score(target[observed], prediction,
                             sample_weight=weights[observed], zero_division=np.nan),
                         'observed_weight_pct': 100 * weights[observed].sum() / weights[test].sum()})
    return pd.DataFrame(rows)


In [ ]:
# The original raw combined-harm variables are already present in this dataset.
Fidelity = SAVED / 'explanatory_fidelity/9f96567a79299de8'
Shallow = SAVED / 'explanatory_fidelity/shallow_8ef4c1c5cfd3b7e6'
display(pd.read_csv(Shallow / 'burden_depths_1_2_3_5_summary.csv').round(3))
display(pd.read_csv(Fidelity / 'one_feature_summary.csv').round(3))
if RUN_EXPLANATIONS:
    for name in ['B', 'E']:
        burden_folds(data, partition_labels[name], weights, WORRY, HARM, [1, 2, 3, 5], 2023).to_csv(
            OUT / f'week04_{name}_burden_folds.csv', index=False)
        for view, features in [('separate_20', FEATURES), ('combined_13', WORRY + HARM)]:
            one_feature_folds(data, partition_labels[name], weights, features).to_csv(
                OUT / f'week04_{name}_{view}_single_feature_folds.csv', index=False)


### 4.4. Read the profiles in survey units

    Select the encoding/allocation/k in the saved tables rather than relying on
    a cluster number having the same meaning across partitions. The frozen B/E
    names below belong to their fixed cluster IDs.

In [ ]:
frozen = joblib.load(ART / 'week05_frozen_objective1_v1/123d248adda6115f/models.joblib')
for name, model in frozen['models'].items():
    print(name, model['spec'])
    display(pd.Series(model['profile_names'], name='profile_name').rename_axis('cluster'))
display(pd.read_csv(SAVED / 'feature_contrasts_all_candidates.csv').head(30))


### 4.5. Decision and limits

    Keep B as the compact four-profile summary and E as the eight-profile view.
    A is coarser and strongly worry-led. C/D are intermediate resolutions of the
    0/1/3 worry-emphasized structure; E is its finest supported resolution. F
    changes the middle worry category more strongly and adds no equally useful
    independent profile interpretation. These are substantive/redundancy choices,
    not claims that A/C/D/F failed ARI.

    B/E are not exactly nested. Burden trees reconstruct B only moderately and E
    poorly (depth-2 balanced accuracy about .588 and .297). Some one-feature
    rules have high recall but much lower precision. Unlike the near-definitional
    Week 3 water-harm cluster, these checks do not reduce B/E to a single original
    feature. No new automatic rejection threshold is introduced after inspection.

## Verification of the combined notebook

The combined notebook is executed in a fresh kernel with refits off. Its tables,
figures and printed results are compared with the three preceding notebooks.
No full stability search is repeated. Frozen data, model caches, assignments,
Week 3, Week 5, Week 6 and the GitHub publication copy are not modified.
See the [merge check](../research/REPRODUCING.md).

## Continue in Week 5

Week 4 ends with its original robustness experiments and provisional B/E
selection. The later optimization of worry spacing $t$ and worry–harm allocation
$\alpha$, fresh confirmation, profile-level review and T5 selection belong to
the [Week 5 workflow](week05_validate_frozen_profiles.ipynb), Sections 1–4.
T5 freezing and secondary transfer assessment follow there in Section 5.
No model, assignment or numerical result was changed by moving these sections.
